# OpenDecision · Phase 2B
## Qwen3.5-4B frozen decision heads, calibration and optional LoRA

**A runnable experiment, not a claim that the model is already accurate.** This notebook extends your working
[Phase 2 probe](https://colab.research.google.com/drive/1O4lxV8DNmDdFNb3fUq10v4efjJ6KQai5), preserving the model ID,
Colab-secret authentication, hidden-state approach and candidate pooling methods. It leaves that notebook unchanged.

### What this run answers
Does a frozen Qwen text backbone support useful **three-way NLI** decisions? Which of last-token, masked mean,
masked max and learned attention pooling works best? Does a linear head suffice, or does an MLP help?
How do held-out calibration, latency and GPU memory change? With the optional switch, does last-block LoRA help?

### Run instructions
Use a **fresh GPU runtime**. Your source run used an **NVIDIA L4**; the default keeps one unquantized model on GPU.
Add a Colab secret named **`HF_TOKEN`** and grant this notebook access. Read the settings cell, then choose **Run all**.
The default `standard` preset uses 2,400 training pairs and separate 300/300/600/600 dev/calibration/matched/mismatched sets.
Use `quick` for a smaller experiment or `smoke` only for plumbing checks. **LoRA is off by default.**

The final cells print a copyable summary, save JSON/CSV/plots/head weights, and optionally copy the results archive into Drive.
A Drive authorization prompt at the end is expected when `copy_results_to_drive=True`.

**Validation at creation:** notebook format/syntax, pooling/masking, metrics, calibration, cache reuse, head training,
and report/export control flow passed CPU tests with synthetic inputs. Those synthetic results are not included here.
The complete model download, CUDA extraction/training and resulting benchmark charts have **not** been executed by the notebook author.
Run the notebook in Colab to obtain actual measurements; no benchmark numbers are prefilled.


### Patch 2b.1.1
Converts loading-diagnostic sets into JSON arrays before building the architecture report,
and replaces the file writer's blanket `default=str` fallback with explicit metadata conversion.
Also includes the native-BF16 hardware check discussed for the T4.
Training settings and experiment selection are unchanged.
The serialization fix was CPU-tested; the full Qwen CUDA experiment has not been run by the notebook author.


## What changes from Phase 2A—and why

| Source notebook behavior | Phase 2B change |
|---|---|
| `AutoModel` loaded a multimodal wrapper, while `AutoModelForCausalLM` loaded a different, smaller object | Reuse the successful causal loader, assert the actual text-only classes, audit components, then retain `.model` |
| Both multi-billion-parameter objects stayed resident | Only one text model is loaded; generation benchmarks run before its wrapper is removed |
| Last/mean/max printed tensor shapes | Train/evaluate linear and MLP probes for each; add learned attention pooling |
| Head memorized two examples | Real MultiNLI examples, held-out tests, premise-group leakage checks and explicit label semantics |
| Unmasked pooling | All pooling methods ignore padding; last pooling finds the last **valid** position |
| One unsynchronized timing per path | Warmups, CUDA synchronization, repeat distributions, generated-token counts and scoped memory peaks |
| Summary hard-coded `removed_generation=True` | Audit actual model classes, tied storage, parameter deltas and implemented/unimplemented capabilities |

**Documented correction:** a tied output projection shares storage with the input token embedding, so removing its module
need not remove any text parameters. The earlier 4.539B versus 4.206B counts compare different loaded graphs; they are not a
measurement of LM-head savings. The new audit does not load the vision tower just to repeat that comparison.

**Scope:** a fixed NLI probe does **not** establish zero-shot arbitrary choice/rubric following, Jev calibration parity,
shared-state branch reuse or Rust/Metal performance. Those remain separate experiments.

### Primary references checked while building this extension
- [Official checkpoint configuration](https://huggingface.co/Qwen/Qwen3.5-4B-Base/blob/main/config.json): actual 4B dimensions and tying.
- [Transformers Qwen3.5 documentation](https://huggingface.co/docs/transformers/v5.17.0/model_doc/qwen3_5): text-only versus multimodal classes and hybrid layers.
- [Pinned Transformers implementation](https://github.com/huggingface/transformers/blob/v5.17.0/src/transformers/models/qwen3_5/modeling_qwen3_5.py): `.model`, output projection and forward interface.
- [MultiNLI dataset card](https://huggingface.co/datasets/nyu-mll/multi_nli): labels, matched/mismatched splits, source and licensing information.
- [PEFT custom-model guidance](https://huggingface.co/docs/peft/developer_guides/custom_models): inspect actual target modules rather than assuming a Llama module layout.
- [Guo et al., On Calibration of Modern Neural Networks](https://proceedings.mlr.press/v70/guo17a.html): held-out temperature scaling.

The source notebook is the provenance for the prior run; these external references support the explicitly listed implementation changes.


## 1 · Install the experiment environment

Transformers is pinned to **5.17.0**. Other packages have bounded version ranges, and resolved versions are recorded.
The install cell does not request a new PyTorch/CUDA build. If you change installed packages after importing them, restart the runtime.
Optional fused DeltaNet/causal-convolution packages are not installed automatically; these are functional-reference timings, not optimized-server claims.

In [ ]:
# Run this cell in a fresh GPU runtime before importing Transformers.
import sys, subprocess, importlib.metadata as im
REQUIRED_TRANSFORMERS = "5.17.0"
if "transformers" in sys.modules:
    loaded = getattr(sys.modules["transformers"], "__version__", "unknown")
    if loaded != REQUIRED_TRANSFORMERS:
        raise RuntimeError(f"Transformers {loaded} is already imported. Restart the runtime, then run all cells.")
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    f"transformers=={REQUIRED_TRANSFORMERS}", "accelerate>=1.2,<2",
    "datasets>=3,<6", "peft>=0.17,<1", "scikit-learn>=1.5,<2",
    "safetensors>=0.5,<1", "matplotlib>=3.8,<4", "scipy>=1.13,<2", "tqdm>=4.66,<5",
])
print("Dependencies installed. The existing Colab PyTorch/CUDA build was not explicitly replaced.")


## 2 · Choose the experiment budget

Change settings here before running the remaining cells.
`run_attention_pooling=True` stores token activations on **local disk**, not GPU; this can use several GiB.
Feature caches are fingerprinted and can be reused within the same runtime. `head_seeds=(17,29,43)` adds seed replication.
LoRA adapts actual linear projections in the last eight hybrid blocks, and is opt-in.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import os, sys, json, time, gc, math, copy, hashlib, platform, random, re, shutil
from collections import Counter
from dataclasses import dataclass, asdict
import importlib.metadata as im
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from scipy.optimize import minimize_scalar
from scipy.special import logsumexp
from tqdm.auto import tqdm
from IPython.display import display, Markdown

@dataclass
class Settings:
    model_id: str = "Qwen/Qwen3.5-4B-Base"
    model_revision: str = "main"   # Resolved once to an immutable commit and recorded.
    dataset_id: str = "nyu-mll/multi_nli"
    dataset_revision: str = "main"
    preset: str = "standard"      # "smoke", "quick", or "standard"; smoke is not evidence.
    seed: int = 17
    head_seeds: tuple = (17,)      # For repeatability estimates: (17, 29, 43).
    max_length: int = 256
    feature_batch_size: int = 2   # Conservative hybrid-attention prefill default.
    head_batch_size: int = 64
    attention_batch_size: int = 16
    head_epochs: int = 20
    head_patience: int = 4
    head_lr: float = 3e-4
    head_weight_decay: float = 1e-3
    mlp_width: int = 256
    run_attention_pooling: bool = True
    run_generation_benchmark: bool = True
    generation_budgets: tuple = (1, 8, 32)
    generation_repeats: int = 5
    benchmark_repeats: int = 15
    benchmark_warmups: int = 2
    benchmark_batch_sizes: tuple = (1, 4, 8)
    run_lora: bool = False        # Optional second stage, not part of the default GPU run.
    lora_last_layers: int = 8
    lora_rank: int = 8
    lora_epochs: int = 2
    lora_batch_size: int = 1
    lora_accumulation: int = 8
    lora_lr: float = 1e-4
    copy_results_to_drive: bool = True   # Colab asks you to authorize a Drive mount at the end.

CFG = Settings()
if not CFG.head_seeds or len(set(CFG.head_seeds)) != len(CFG.head_seeds):
    raise ValueError("head_seeds must contain at least one unique integer seed.")
if min(CFG.feature_batch_size, CFG.head_batch_size, CFG.attention_batch_size, CFG.head_epochs, CFG.head_patience, CFG.benchmark_repeats, CFG.generation_repeats) < 1:
    raise ValueError("Batch sizes, repeat counts, epochs and patience must be positive.")
if CFG.run_generation_benchmark and (not CFG.generation_budgets or min(CFG.generation_budgets) < 1):
    raise ValueError("Generation benchmark needs positive token budgets.")
SIZES = {
    "smoke": {"train": 60, "dev": 30, "calibration": 30, "test_matched": 30, "test_mismatched": 30},
    "quick": {"train": 600, "dev": 150, "calibration": 150, "test_matched": 300, "test_mismatched": 300},
    "standard": {"train": 2400, "dev": 300, "calibration": 300, "test_matched": 600, "test_mismatched": 600},
}[CFG.preset]
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
ROOT = Path("/content/opendecision_phase2b") if Path("/content").exists() else Path.cwd() / "opendecision_phase2b_runs"
OUT = ROOT / RUN_ID
CACHE_ROOT = ROOT / "feature_cache"
OUT.mkdir(parents=True, exist_ok=True)
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
NOTEBOOK_VERSION = "2b.1.1"
SOURCE_NOTEBOOK = {
    "name": "Copy of OpenDecision_Phase2_Qwen3_5_4B_Probe.ipynb",
    "drive_id": "1O4lxV8DNmDdFNb3fUq10v4efjJ6KQai5",
    "url": "https://colab.research.google.com/drive/1O4lxV8DNmDdFNb3fUq10v4efjJ6KQai5",
    "observations_not_benchmark_claims": {
        "multimodal_backbone_parameters": 4539265536,
        "causal_lm_parameters": 4205751296,
        "single_generation_seconds": 7.480607748031616,
        "single_decision_seconds": 0.2616415023803711,
        "head_training_examples": 2,
        "gpu": "NVIDIA L4",
    },
}
STATUS = {"frozen": "not_run", "lora": "disabled" if not CFG.run_lora else "not_run"}
WARNINGS = []

def json_ready(value):
    """Normalize audit metadata without converting numbers or arrays into strings."""
    if isinstance(value, dict):
        return {key: json_ready(item) for key, item in value.items()}
    if isinstance(value, (set, frozenset)):
        return [json_ready(item) for item in sorted(value, key=str)]
    if isinstance(value, (list, tuple)):
        return [json_ready(item) for item in value]
    if isinstance(value, np.ndarray):
        return json_ready(value.tolist())
    if isinstance(value, np.generic):
        return json_ready(value.item())
    if isinstance(value, (Path, torch.dtype, torch.device)):
        return str(value)
    return value  # Unsupported objects still raise; do not silently stringify them.

def json_write(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    payload = json.dumps(json_ready(value), indent=2, allow_nan=False)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(payload, encoding="utf-8")
    tmp.replace(path)

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(CFG.seed)
print(json.dumps({"configuration": asdict(CFG), "requested_split_sizes": SIZES, "output": str(OUT)}, indent=2))
if CFG.preset == "smoke":
    WARNINGS.append("Smoke preset: validates plumbing only; do not infer model quality from these sample sizes.")


## 3 · Authenticate and record the exact runtime

This cell reads the secret without printing it. It rejects CPU execution and a runtime already holding large GPU models,
resolves model/dataset revisions to commits, and records versions, precision and GPU identity.
No API is called to upload a model or dataset to Hugging Face.

In [ ]:
# Credential is read, never printed, persisted in reports, or embedded into code.
HF_TOKEN = os.environ.get("HF_TOKEN")
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN") or HF_TOKEN
except Exception:
    pass
if not HF_TOKEN:
    print("HF_TOKEN not found. Trying public access; a Colab secret named HF_TOKEN is recommended.")
if not torch.cuda.is_available():
    raise RuntimeError("Choose Runtime > Change runtime type > GPU, then restart and run all cells.")
DEVICE = torch.device("cuda:0")
if torch.cuda.memory_allocated(DEVICE) > 512 * 1024**2:
    raise RuntimeError("Existing GPU tensors/models detected. Use a fresh runtime for uncontaminated memory measurements.")
with torch.cuda.device(DEVICE):
    NATIVE_BF16 = torch.cuda.is_bf16_supported(including_emulation=False)
COMPUTE_CAPABILITY = torch.cuda.get_device_capability(DEVICE)
DTYPE = torch.bfloat16 if NATIVE_BF16 else torch.float16
if DTYPE == torch.float16:
    WARNINGS.append("FP16 fallback selected. Check numerical-finiteness diagnostics; do not mix its timings with BF16 results.")
free_bytes, total_bytes = torch.cuda.mem_get_info(DEVICE)
if free_bytes < 11 * 1024**3:
    raise RuntimeError("Less than 11 GiB of GPU memory is free. This unquantized experiment needs a fresh, larger GPU.")
ENV = {
    "python": sys.version.split()[0], "platform": platform.platform(),
    "torch": torch.__version__, "cuda_runtime": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(DEVICE), "gpu_total_gib": total_bytes / 1024**3,
    "initial_gpu_free_gib": free_bytes / 1024**3, "dtype": str(DTYPE),
    "compute_capability": list(COMPUTE_CAPABILITY), "native_bf16": NATIVE_BF16,
    "packages": {p: im.version(p) for p in ("transformers", "accelerate", "datasets", "peft", "scikit-learn", "safetensors", "numpy", "scipy")},
}
# Package names/versions only; do not dump environment variables or authenticated package URLs.
json_write(OUT / "environment.json", ENV)
print(json.dumps(ENV, indent=2))

from huggingface_hub import HfApi
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig
api = HfApi(token=HF_TOKEN)
MODEL_SHA = api.model_info(CFG.model_id, revision=CFG.model_revision).sha
DATA_SHA = api.dataset_info(CFG.dataset_id, revision=CFG.dataset_revision).sha
PROVENANCE = {"source_notebook": SOURCE_NOTEBOOK, "model_commit": MODEL_SHA, "dataset_commit": DATA_SHA,
              "notebook_version": NOTEBOOK_VERSION, "settings": asdict(CFG)}
json_write(OUT / "provenance.json", PROVENANCE)
print("Resolved model:", MODEL_SHA, "\nResolved dataset:", DATA_SHA)


## 4 · Build leakage-checked NLI partitions

Train uses the official training split. Dev, calibration and matched test use disjoint **premise-hash buckets** from
`validation_matched`; mismatched evaluation uses `validation_mismatched`. Invalid labels and repeated sentence pairs are excluded.
The sample sizes preserve the sampled source class mixture rather than manufacturing equal class priors.

Dev selects architecture/epochs. Calibration fits temperature. Neither test set is used for either choice.
The base model may have seen benchmark examples during pretraining; this notebook cannot establish decontamination.

In [ ]:
from datasets import load_dataset
raw = load_dataset(CFG.dataset_id, revision=DATA_SHA, token=HF_TOKEN)
LABELS = list(raw["train"].features["label"].names)
if set(LABELS) != {"entailment", "neutral", "contradiction"}:
    raise RuntimeError(f"Unexpected NLI labels: {LABELS}")
N_CLASSES = len(LABELS)

# Use a normalized premise hash for grouping. No premise may cross our local splits.
def normalized_text(text):
    return " ".join(text.split()).casefold()

def text_hash(text):
    return hashlib.sha256(normalized_text(text).encode("utf-8")).hexdigest()

def normalize_row(row, source_split, index):
    p, h = row["premise"].strip(), row["hypothesis"].strip()
    y = int(row["label"])
    if not p or not h or not (0 <= y < N_CLASSES):
        return None
    return {"premise": p, "hypothesis": h, "label": y, "genre": row.get("genre", "unknown"),
            "group": text_hash(p), "pair_hash": text_hash(p + "\n<HYPOTHESIS>\n" + h),
            "source_split": source_split, "source_index": int(index), "pair_id": str(row.get("pairID", index))}

def collect_partition(dataset, source_name, limit, salt, blocked_groups, bucket=None):
    chosen, seen_pairs = [], set()
    for idx in np.random.default_rng(salt).permutation(len(dataset)):
        row = normalize_row(dataset[int(idx)], source_name, int(idx))
        if row is None or row["group"] in blocked_groups or row["pair_hash"] in seen_pairs:
            continue
        # Assign every premise deterministically to just one matched-validation partition.
        if bucket is not None and int(row["group"][:16], 16) % 3 != bucket:
            continue
        chosen.append(row); seen_pairs.add(row["pair_hash"])
        if len(chosen) >= limit:
            break
    if len(chosen) < limit:
        raise RuntimeError(f"Only found {len(chosen)} eligible rows for {source_name}; requested {limit}.")
    return chosen

SPLITS = {}
used_groups = set()
SPLITS["train"] = collect_partition(raw["train"], "train", SIZES["train"], CFG.seed, used_groups)
used_groups.update(r["group"] for r in SPLITS["train"])
for j, split in enumerate(("dev", "calibration", "test_matched")):
    SPLITS[split] = collect_partition(raw["validation_matched"], "validation_matched", SIZES[split], CFG.seed + j + 1,
                                       used_groups, bucket=j)
    used_groups.update(r["group"] for r in SPLITS[split])
SPLITS["test_mismatched"] = collect_partition(raw["validation_mismatched"], "validation_mismatched", SIZES["test_mismatched"],
                                               CFG.seed + 4, used_groups)

def audit_splits(splits):
    overlaps = {}
    names = list(splits)
    for i, a in enumerate(names):
        for b in names[i + 1:]:
            n = len({r["group"] for r in splits[a]} & {r["group"] for r in splits[b]})
            overlaps[f"{a}__{b}"] = n
            if n:
                raise ValueError(f"Premise leakage between {a} and {b}: {n} groups")
    return overlaps

DATA_AUDIT = {"premise_group_overlaps": audit_splits(SPLITS), "label_order": LABELS, "splits": {}}
for name, rows in SPLITS.items():
    counts = Counter(LABELS[r["label"]] for r in rows)
    if len(counts) != N_CLASSES:
        raise RuntimeError(f"Missing a class in {name}; increase sample size/change seed.")
    DATA_AUDIT["splits"][name] = {"n": len(rows), "class_counts": dict(counts),
                                   "genres": dict(Counter(r["genre"] for r in rows)),
                                   "fingerprint": hashlib.sha256("".join(r["pair_hash"] for r in rows).encode()).hexdigest()}
json_write(OUT / "data_audit.json", DATA_AUDIT)
json_write(OUT / "split_manifest.json", {s: [{k: v for k, v in r.items() if k not in ("premise", "hypothesis")} for r in rows]
                                        for s, rows in SPLITS.items()})
display(pd.DataFrame([{ "split": s, "examples": len(rows), **dict(Counter(LABELS[r["label"]] for r in rows)) }
                       for s, rows in SPLITS.items()]))
print("Verified: no normalized premise appears in two local splits. Base-model pretraining contamination is unknown.")
del raw; gc.collect()


## 5 · Freeze the prompt and tokenization contract

The model sees premise, hypothesis and **descriptions of all three NLI labels**, never the correct answer.
Labels come from dataset metadata. Truncation keeps the rubric and prioritizes the hypothesis, with counts reported.
If many inputs are truncated, rerun with a larger `max_length`; results from different limits are separate experiments.
Segment-by-segment tokenization is intentional and is exported for eventual Rust parity.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CFG.model_id, revision=MODEL_SHA, token=HF_TOKEN, trust_remote_code=False)
if tokenizer.pad_token_id is None:
    if tokenizer.eos_token_id is None:
        raise RuntimeError("Tokenizer has neither a pad token nor an EOS token.")
    tokenizer.pad_token = tokenizer.eos_token
# Right padding; pooling and generation benchmarks account for it explicitly.
tokenizer.padding_side = "right"
PROMPT_VERSION = "nli-described-abc-v1"
DESCRIPTIONS = {
    "entailment": "The premise makes the hypothesis true.",
    "neutral": "The premise does not establish whether the hypothesis is true or false.",
    "contradiction": "The premise makes the hypothesis false.",
}
CODES = [chr(ord("A") + i) for i in range(N_CLASSES)]
PREFIX = "Decide the relationship between the premise and hypothesis.\nPremise:\n"
RUBRIC = "\nChoices:\n" + "\n".join(f"{c}: {label}. {DESCRIPTIONS[label]}" for c, label in zip(CODES, LABELS))
TAIL = RUBRIC + "\nReturn the choice code.\nAnswer:"

def enc(text):
    return tokenizer.encode(text, add_special_tokens=False)

def encode_record(row):
    # Explicit token-segment contract preserves the rubric; it is also exported for Rust.
    prefix, hyp_prefix, tail = enc(PREFIX), enc("\nHypothesis:\n"), enc(TAIL)
    p, h = enc(row["premise"]), enc(row["hypothesis"])
    budget = CFG.max_length - len(prefix) - len(hyp_prefix) - len(tail)
    if budget < 32:
        raise ValueError("max_length leaves too few tokens for premise/hypothesis.")
    h_kept = h[:max(1, min(len(h), budget - min(24, budget // 3)))]
    p_kept = p[:budget - len(h_kept)]
    ids = prefix + p_kept + hyp_prefix + h_kept + tail
    return {**row, "input_ids": ids, "length": len(ids), "premise_truncated": len(p_kept) < len(p),
            "hypothesis_truncated": len(h_kept) < len(h), "original_content_tokens": len(p) + len(h)}

TOKENS = {split: [encode_record(r) for r in rows] for split, rows in SPLITS.items()}
TOKEN_STATS = {s: {"mean_tokens": float(np.mean([r["length"] for r in rows])),
                   "max_tokens": max(r["length"] for r in rows),
                   "premise_truncated": sum(r["premise_truncated"] for r in rows),
                   "hypothesis_truncated": sum(r["hypothesis_truncated"] for r in rows)} for s, rows in TOKENS.items()}
json_write(OUT / "token_stats.json", TOKEN_STATS)
for s, v in TOKEN_STATS.items():
    if v["premise_truncated"] or v["hypothesis_truncated"]:
        WARNINGS.append(f"{s}: truncated {v['premise_truncated']} premises and {v['hypothesis_truncated']} hypotheses.")
display(pd.DataFrame(TOKEN_STATS).T)
print("Prompt sample (label not supplied to the model):\n", tokenizer.decode(TOKENS["train"][0]["input_ids"]))

class TokenRows(Dataset):
    def __init__(self, rows): self.rows = rows
    def __len__(self): return len(self.rows)
    def __getitem__(self, idx): return self.rows[idx]

def collate_tokens(rows):
    n = max(len(r["input_ids"]) for r in rows)
    ids = torch.full((len(rows), n), tokenizer.pad_token_id, dtype=torch.long)
    mask = torch.zeros_like(ids)
    for i, r in enumerate(rows):
        length = len(r["input_ids"])
        ids[i, :length] = torch.tensor(r["input_ids"], dtype=torch.long)
        mask[i, :length] = 1
    return {"input_ids": ids, "attention_mask": mask, "labels": torch.tensor([r["label"] for r in rows])}

def model_inputs(batch):
    return {k: v.to(DEVICE) for k, v in batch.items() if k in ("input_ids", "attention_mask")}


## 6 · Load one text model and audit what is removable

Use the loader that succeeded in your saved notebook. Before doing research, assert the actual classes and fail on
missing/mismatched text weights. Count unique parameter objects and storage; distinguish the logical LM-head size from
its **marginal** parameter cost. A tiny copied vocabulary slice supports an untuned A/B/C baseline later.

In [ ]:
def gpu_memory():
    return {"allocated_mib": torch.cuda.memory_allocated(DEVICE) / 1024**2,
            "reserved_mib": torch.cuda.memory_reserved(DEVICE) / 1024**2,
            "peak_allocated_mib": torch.cuda.max_memory_allocated(DEVICE) / 1024**2}

def unique_parameter_stats(module):
    params = list(module.parameters())  # PyTorch deduplicates shared Parameter objects.
    storage = {}
    for p in params:
        s = p.untyped_storage()
        storage[(str(p.device), s.data_ptr())] = s.nbytes()
    return {"parameters": sum(p.numel() for p in params),
            "unique_parameter_storage_mib": sum(storage.values()) / 1024**2,
            "trainable_parameters": sum(p.numel() for p in params if p.requires_grad)}

print("Loading ONE text-only causal model, using the loader that succeeded in the original notebook.")
torch.cuda.reset_peak_memory_stats(DEVICE)
lm, load_info = AutoModelForCausalLM.from_pretrained(
    CFG.model_id, revision=MODEL_SHA, token=HF_TOKEN, trust_remote_code=False,
    dtype=DTYPE, device_map={"": 0}, attn_implementation="sdpa", output_loading_info=True,
)
# Loading diagnostics may contain sets in Transformers 5.x.
load_info = json_ready(load_info)
lm.eval()
if type(lm).__name__ != "Qwen3_5ForCausalLM" or type(lm.model).__name__ != "Qwen3_5TextModel":
    raise RuntimeError(f"Unexpected loaded classes: {type(lm).__name__}, {type(lm.model).__name__}. Do not compare sizes blindly.")
if any("visual" in n for n, _ in lm.named_modules()):
    raise RuntimeError("A vision tower was loaded: aborting the text-only experiment.")
if load_info.get("missing_keys") or load_info.get("mismatched_keys") or load_info.get("error_msgs"):
    json_write(OUT / "load_info_failure.json", load_info)
    raise RuntimeError("Missing/mismatched model weights. Inspect load_info_failure.json; do not train a partially initialized backbone.")
text_config = getattr(lm.config, "text_config", lm.config)
HIDDEN_SIZE = int(text_config.hidden_size)
in_weight = lm.get_input_embeddings().weight
out_weight = lm.get_output_embeddings().weight
TIED = in_weight.data_ptr() == out_weight.data_ptr()
full_stats = unique_parameter_stats(lm)
backbone_stats = unique_parameter_stats(lm.model)
ARCH = {"loaded_class": type(lm).__name__, "backbone_class": type(lm.model).__name__,
        "hidden_size": HIDDEN_SIZE, "layers": int(text_config.num_hidden_layers),
        "vocab_size": int(text_config.vocab_size), "input_output_weights_tied": TIED,
        "layer_types": list(getattr(text_config, "layer_types", [])),
        "logical_lm_head_parameters": out_weight.numel(),
        "text_lm": full_stats, "text_backbone": backbone_stats,
        "marginal_lm_head_parameters": full_stats["parameters"] - backbone_stats["parameters"],
        "gpu_after_single_model_load": gpu_memory(), "missing_keys": load_info.get("missing_keys", []),
        "unexpected_key_count": len(load_info.get("unexpected_keys", [])),
        "vision_tower_loaded": False, "quantized": False, "attention_implementation_requested": "sdpa"}
json_write(OUT / "architecture.json", ARCH)
json_write(OUT / "text_config.json", text_config.to_dict())
json_write(OUT / "loading_info.json", load_info)
print(json.dumps(ARCH, indent=2))
# Tiny immutable candidate-vocabulary projection for an untuned finite-code baseline.
CODE_TOKEN_IDS = None
for prefix in (" ", ""):
    candidate_ids = [enc(prefix + code) for code in CODES]
    if all(len(ids) == 1 for ids in candidate_ids):
        CODE_TOKEN_IDS = [ids[0] for ids in candidate_ids]
        CODE_TOKEN_PREFIX = prefix
        break
CODE_WEIGHTS = out_weight[CODE_TOKEN_IDS].detach().float().cpu().clone() if CODE_TOKEN_IDS else None
# Do not keep a separate reference to the full output matrix after detaching the backbone.
del in_weight, out_weight


## 7 · Measure generation before removing the wrapper

This benchmark uses one fixed dev prompt and unpadded, GPU-resident inputs. Generation uses greedy decoding with budgets
of 1, 8 and 32 new tokens. EOS may stop earlier, so **actual token counts** are saved. No minimum token count is forced.
Timing excludes tokenization and text decoding. These are workload comparisons, not evidence of equal prediction quality.

In [ ]:
def timed_calls(fn, repeats, warmups=2):
    for _ in range(warmups):
        value = fn()
        del value
    torch.cuda.synchronize(DEVICE)
    baseline = torch.cuda.memory_allocated(DEVICE)
    torch.cuda.reset_peak_memory_stats(DEVICE)
    elapsed = []
    for _ in range(repeats):
        torch.cuda.synchronize(DEVICE)
        start = time.perf_counter()
        value = fn()
        torch.cuda.synchronize(DEVICE)
        elapsed.append((time.perf_counter() - start) * 1000)
        del value
    return {"repeats": repeats, "warmups": warmups, "p50_ms": float(np.median(elapsed)),
            "p95_ms": float(np.percentile(elapsed, 95)), "mean_ms": float(np.mean(elapsed)),
            "min_ms": float(min(elapsed)), "raw_ms": elapsed,
            "resident_allocated_mib": baseline / 1024**2,
            "peak_allocated_mib": torch.cuda.max_memory_allocated(DEVICE) / 1024**2,
            "peak_extra_allocated_mib": (torch.cuda.max_memory_allocated(DEVICE) - baseline) / 1024**2}

GENERATION_RESULTS = []
# One fixed, unpadded prompt for matched workload timing; no test data used for tuning.
BENCH_ROW = TOKENS["dev"][0]
bench_batch = model_inputs(collate_tokens([BENCH_ROW]))
if CFG.run_generation_benchmark:
    for budget in CFG.generation_budgets:
        token_counts = []
        @torch.inference_mode()
        def generate_once():
            seq = lm.generate(**bench_batch, do_sample=False, num_beams=1, use_cache=True,
                              max_new_tokens=int(budget), pad_token_id=tokenizer.pad_token_id,
                              return_dict_in_generate=False)
            token_counts.append(int(seq.shape[1] - bench_batch["input_ids"].shape[1]))
            return seq
        result = timed_calls(generate_once, CFG.generation_repeats, CFG.benchmark_warmups)
        measured_counts = token_counts[-CFG.generation_repeats:]
        result.update({"mode": "unconstrained_greedy_generation", "max_new_tokens": int(budget),
                       "actual_new_tokens": measured_counts, "input_tokens": BENCH_ROW["length"],
                       "batch_size": 1, "scope": "GPU-resident input to generated IDs; excludes tokenization and text decoding"})
        GENERATION_RESULTS.append(result)
        print(f"Generation budget {budget}: {result['p50_ms']:.2f} ms median; actual tokens {measured_counts}")
    del generate_once
json_write(OUT / "generation_benchmark.json", GENERATION_RESULTS)
# Each benchmark starts without output/cache references retained by the notebook.
del bench_batch
gc.collect(); torch.cuda.empty_cache()


## 8 · Keep only the text backbone and validate batching

Retain the text model object; discard the generation wrapper. No new 4B model is instantiated.
Disable caches and all-layer hidden-state capture for this full-sequence feature experiment. This is **not** an implementation
of reusable prefix state. Check single-example versus padded-batch hidden values before extracting features.

In [ ]:
# This is the same set of text weights, not a second model allocation.
backbone = lm.model
backbone.config.use_cache = False
backbone.eval()
for p in backbone.parameters():
    p.requires_grad_(False)
del lm
# In a tied model the token embedding remains necessary for input token lookup.
gc.collect(); torch.cuda.empty_cache(); torch.cuda.synchronize(DEVICE)
ARCH["backbone_after_detach"] = unique_parameter_stats(backbone)
ARCH["gpu_after_detach"] = gpu_memory()
ARCH["decision_path"] = "Qwen3_5TextModel -> mask-aware pooling -> learned head -> calibrated softmax"
ARCH["generative_decode_used_in_decision_path"] = False
ARCH["shared_prefill_branching_implemented"] = False
json_write(OUT / "architecture.json", ARCH)
print("Active backbone:", type(backbone).__name__)
print("Unique parameters retained:", f"{ARCH['backbone_after_detach']['parameters']:,}")
print("Marginal parameters removed with the LM head:", f"{ARCH['marginal_lm_head_parameters']:,}")
print("Allocated MiB:", round(ARCH["gpu_after_detach"]["allocated_mib"], 1))

# Check real padding equivalence before caching an entire dataset.
def forward_hidden(batch, network=None):
    network = backbone if network is None else network
    out = network(**model_inputs(batch), use_cache=False, output_hidden_states=False, return_dict=True)
    return out.last_hidden_state

with torch.no_grad():
    pair = TOKENS["dev"][:2]
    padded = collate_tokens(pair)
    bh = forward_hidden(padded)
    discrepancies = []
    for i, row in enumerate(pair):
        sh = forward_hidden(collate_tokens([row]))
        difference = (sh[0].float() - bh[i, :row["length"]].float()).abs()
        discrepancies.append({"max_abs": difference.max().item(), "mean_abs": difference.mean().item()})
    del sh, bh, difference, padded
ARCH["batch_padding_hidden_differences"] = discrepancies
json_write(OUT / "architecture.json", ARCH)
print("Single vs padded-batch hidden differences:", discrepancies)
print("Low-precision batched kernels can differ slightly. Large changes warrant batch_size=1 before interpreting results.")


## 9 · Define mask-aware probes and run unit checks

The linear probe remains linear after a train-only affine feature standardization. MLP probes add LayerNorm, a 256-wide
hidden layer and dropout. Learned attention pooling trains a token scorer and an MLP, using the same frozen token representations.
Extra parameters differ between methods and are reported; there is no assumption that attention pooling must win.

In [ ]:
def validate_hidden_mask(hidden, mask):
    if hidden.ndim != 3 or mask.shape != hidden.shape[:2]:
        raise ValueError("Expected hidden[B,T,H] and mask[B,T].")
    mask = mask.bool()
    if not mask.any(dim=1).all():
        raise ValueError("Every example needs at least one valid token.")
    return mask

def pool_hidden(hidden, mask, method):
    mask = validate_hidden_mask(hidden, mask)
    x = hidden.float()
    if method == "last":
        positions = torch.arange(mask.shape[1], device=mask.device).expand_as(mask)
        last = positions.masked_fill(~mask, -1).max(dim=1).values
        return x[torch.arange(x.shape[0], device=x.device), last]
    if method == "mean":
        return x.masked_fill(~mask[..., None], 0).sum(dim=1) / mask.sum(dim=1, keepdim=True)
    if method == "max":
        return x.masked_fill(~mask[..., None], -torch.inf).max(dim=1).values
    raise ValueError(f"Unknown pooling method: {method}")

class FixedHead(nn.Module):
    def __init__(self, hidden, classes, architecture="linear", width=256):
        super().__init__()
        self.architecture = architecture
        self.register_buffer("feature_mean", torch.zeros(hidden))
        self.register_buffer("feature_std", torch.ones(hidden))
        # The linear probe is actually linear: no learned nonlinear normalization before it.
        if architecture == "linear":
            self.net = nn.Linear(hidden, classes)
        elif architecture == "mlp":
            self.net = nn.Sequential(nn.LayerNorm(hidden), nn.Linear(hidden, width), nn.GELU(),
                                     nn.Dropout(0.1), nn.Linear(width, classes))
        else:
            raise ValueError(architecture)
    def forward(self, x): return self.net((x.float() - self.feature_mean) / self.feature_std)

class AttentionHead(nn.Module):
    def __init__(self, hidden, classes, width=256):
        super().__init__()
        self.token_norm = nn.LayerNorm(hidden)
        self.score = nn.Linear(hidden, 1, bias=False)
        self.classifier = FixedHead(hidden, classes, "mlp", width)
    def forward(self, hidden, mask):
        mask = validate_hidden_mask(hidden, mask)
        x = hidden.float()
        logits = self.score(self.token_norm(x)).squeeze(-1).masked_fill(~mask, -torch.inf)
        weights = logits.softmax(dim=1)
        pooled = (weights[..., None] * x.masked_fill(~mask[..., None], 0)).sum(dim=1)
        return self.classifier(pooled)

# Structural tests, not a model-quality evaluation.
def test_pooling_primitives():
    x = torch.tensor([[[1., 2.], [3., 4.], [999., 999.]], [[777., 777.], [5., 6.], [7., 8.]]])
    mask = torch.tensor([[1, 1, 0], [0, 1, 1]])
    assert torch.equal(pool_hidden(x, mask, "last"), torch.tensor([[3., 4.], [7., 8.]]))
    assert torch.equal(pool_hidden(x, mask, "mean"), torch.tensor([[2., 3.], [6., 7.]]))
    assert torch.equal(pool_hidden(x, mask, "max"), torch.tensor([[3., 4.], [7., 8.]]))
    try:
        pool_hidden(x, torch.zeros_like(mask), "last")
    except ValueError:
        pass
    else:
        raise AssertionError("Empty masks must be rejected.")
    h = AttentionHead(2, 3, 4).eval()
    with torch.no_grad():
        batched = h(x, mask)
        for i in range(2):
            alone = h(x[i:i+1, mask[i].bool()], torch.ones(1, 2))
            torch.testing.assert_close(alone[0], batched[i], atol=1e-6, rtol=1e-5)
    return "passed"

print("Padding and attention-pooling unit checks:", test_pooling_primitives())


## 10 · Extract features once, not once per training epoch

One frozen forward produces features for all pooling methods. Last/mean/max vectors are small; learned attention uses
ragged FP16 final-token activations in local NumPy memory maps. Padding is excluded, all activations must be finite, and
incomplete caches are not reused. Runtime, prompt, model and dataset fingerprints prevent accidental stale-cache comparisons.

Disk-cache throughput includes I/O. It is **not** reported as production inference throughput.

In [ ]:
# FP16 token activations are memory-mapped on local disk. The 4B trunk runs once per example,
# not once per head epoch. All fixed pools are derived from the same cached precision.
class FeatureStore:
    def __init__(self, directory):
        self.directory = Path(directory)
        self.meta = json.loads((self.directory / "complete.json").read_text())
        self.offsets = np.load(self.directory / "offsets.npy", mmap_mode="r")
        self.labels = np.load(self.directory / "labels.npy", mmap_mode="r")
        self.fixed = {p: np.load(self.directory / f"{p}.npy", mmap_mode="r") for p in ("last", "mean", "max")}
        self.tokens = np.load(self.directory / "tokens.npy", mmap_mode="r") if self.meta["has_tokens"] else None
    def __len__(self): return len(self.labels)
    def get_tokens(self, i):
        if self.tokens is None: raise ValueError("Token cache was not enabled.")
        a, b = int(self.offsets[i]), int(self.offsets[i+1])
        return np.array(self.tokens[a:b], dtype=np.float32, copy=True)

CACHE_TIMES = {}

def extract_features(split, rows):
    fingerprint = {"notebook": NOTEBOOK_VERSION, "model_sha": MODEL_SHA, "dataset_sha": DATA_SHA,
                   "dtype": str(DTYPE), "activation_storage": "float16", "prompt_version": PROMPT_VERSION,
                   "max_length": CFG.max_length, "batch_size": CFG.feature_batch_size,
                   "transformers": ENV["packages"]["transformers"], "torch": ENV["torch"],
                   "gpu": ENV["gpu"], "tokens": CFG.run_attention_pooling,
                   "ids_sha256": hashlib.sha256(json.dumps([r["input_ids"] for r in rows]).encode()).hexdigest(),
                   "labels_sha256": hashlib.sha256(bytes(r["label"] for r in rows)).hexdigest()}
    key = hashlib.sha256(json.dumps(fingerprint, sort_keys=True).encode()).hexdigest()[:24]
    directory = CACHE_ROOT / f"{split}_{key}"
    if (directory / "complete.json").exists():
        store = FeatureStore(directory)
        if store.meta["fingerprint"] != fingerprint:
            raise RuntimeError("Cache fingerprint mismatch.")
        CACHE_TIMES[split] = {"cache_hit": True, "examples": len(rows)}
        return store
    if directory.exists():
        # Only this experiment's unfinished, fingerprinted cache directory is replaced.
        shutil.rmtree(directory)
    directory.mkdir(parents=True)
    offsets = np.concatenate(([0], np.cumsum([r["length"] for r in rows]))).astype(np.int64)
    estimated_bytes = int(offsets[-1]) * HIDDEN_SIZE * 2 if CFG.run_attention_pooling else 0
    if shutil.disk_usage(CACHE_ROOT).free < estimated_bytes + 2 * 1024**3:
        raise RuntimeError("Not enough local disk for activations. Reduce preset/max_length or disable attention pooling.")
    np.save(directory / "offsets.npy", offsets)
    np.save(directory / "labels.npy", np.array([r["label"] for r in rows], dtype=np.int64))
    fixed = {p: np.lib.format.open_memmap(directory / f"{p}.npy", mode="w+", dtype=np.float32,
                                          shape=(len(rows), HIDDEN_SIZE)) for p in ("last", "mean", "max")}
    token_map = np.lib.format.open_memmap(directory / "tokens.npy", mode="w+", dtype=np.float16,
                                          shape=(int(offsets[-1]), HIDDEN_SIZE)) if CFG.run_attention_pooling else None
    started = time.perf_counter()
    loader = DataLoader(TokenRows(rows), batch_size=CFG.feature_batch_size, shuffle=False,
                        num_workers=0, collate_fn=collate_tokens)
    index = 0
    with torch.no_grad():
        for batch in tqdm(loader, desc=f"Frozen features: {split}"):
            hidden = forward_hidden(batch).to(torch.float16)
            if not torch.isfinite(hidden).all():
                raise FloatingPointError("Non-finite cached activations. Retry BF16 backbone on a compatible GPU or inspect load.")
            mask = batch["attention_mask"].to(DEVICE).bool()
            pooled = {p: pool_hidden(hidden, mask, p).cpu().numpy() for p in fixed}
            cpu_tokens = hidden.cpu().numpy() if token_map is not None else None
            size = hidden.shape[0]
            for p in fixed:
                fixed[p][index:index+size] = pooled[p]
            if token_map is not None:
                for local in range(size):
                    start, stop = offsets[index + local:index + local + 2]
                    token_map[int(start):int(stop)] = cpu_tokens[local, :int(stop-start)]
            index += size
            del hidden, mask, pooled, cpu_tokens
    torch.cuda.synchronize(DEVICE)
    for array in fixed.values(): array.flush()
    if token_map is not None: token_map.flush()
    elapsed = time.perf_counter() - started
    meta = {"fingerprint": fingerprint, "examples": len(rows), "total_tokens": int(offsets[-1]),
            "has_tokens": CFG.run_attention_pooling, "hidden_size": HIDDEN_SIZE,
            "activation_bytes": estimated_bytes, "extraction_seconds": elapsed}
    json_write(directory / "complete.json", meta)  # Written last; interrupted caches are not trusted.
    CACHE_TIMES[split] = {"cache_hit": False, "examples": len(rows), "seconds": elapsed,
                           "examples_per_second_including_disk": len(rows) / elapsed}
    return FeatureStore(directory)

STORES = {split: extract_features(split, rows) for split, rows in TOKENS.items()}
json_write(OUT / "feature_extraction.json", CACHE_TIMES)
print("Feature cache ready. No hidden-state cache will be copied into the final results bundle.")


## 11 · Define probability metrics and calibration

Report accuracy, macro F1, negative log-likelihood, multiclass Brier score (sum over classes) and 15-bin top-label ECE.
Temperature is fit on the **calibration split only**. Test accuracy intervals use a premise-cluster bootstrap.
ECE is descriptive and bin-dependent; no model is selected by ECE alone.

In [ ]:
def probabilities(logits, temperature=1.0):
    z = np.asarray(logits, dtype=np.float64) / float(temperature)
    if z.ndim != 2 or not np.isfinite(z).all() or temperature <= 0:
        raise ValueError("Expected finite logits[N,K] and a positive temperature.")
    return np.exp(z - logsumexp(z, axis=1, keepdims=True))

def reliability_bins(probs, y, bins=15):
    confidence = probs.max(axis=1)
    correct = probs.argmax(axis=1) == y
    assignments = np.minimum((confidence * bins).astype(int), bins - 1)
    rows = []
    for b in range(bins):
        keep = assignments == b
        rows.append({"bin": b, "lower": b / bins, "upper": (b + 1) / bins,
                     "count": int(keep.sum()), "confidence": float(confidence[keep].mean()) if keep.any() else None,
                     "accuracy": float(correct[keep].mean()) if keep.any() else None})
    return rows

def metric_values(logits, y, temperature=1.0):
    y = np.asarray(y, dtype=np.int64)
    probs = probabilities(logits, temperature)
    if len(y) != len(probs) or not len(y) or (y < 0).any() or (y >= probs.shape[1]).any():
        raise ValueError("Invalid labels or an empty evaluation set.")
    scaled = np.asarray(logits, np.float64) / temperature
    nll = float(np.mean(logsumexp(scaled, axis=1) - scaled[np.arange(len(y)), y]))
    brier = float(np.mean(np.sum((probs - np.eye(probs.shape[1])[y])**2, axis=1)))
    bins = reliability_bins(probs, y)
    ece = sum(r["count"] * abs(r["accuracy"] - r["confidence"]) for r in bins if r["count"]) / len(y)
    return {"n": len(y), "accuracy": float(accuracy_score(y, probs.argmax(axis=1))),
            "macro_f1": float(f1_score(y, probs.argmax(axis=1), average="macro", labels=list(range(probs.shape[1])), zero_division=0)),
            "nll": nll, "brier_sum_classes": brier, "ece_toplabel_15bins": float(ece),
            "temperature": float(temperature)}

def fit_temperature(logits, y):
    logits, y = np.asarray(logits, np.float64), np.asarray(y, np.int64)
    def objective(log_t):
        scaled = logits / np.exp(log_t)
        return float(np.mean(logsumexp(scaled, axis=1) - scaled[np.arange(len(y)), y]))
    result = minimize_scalar(objective, bounds=(np.log(0.05), np.log(20.0)), method="bounded")
    if not result.success or not np.isfinite(result.fun):
        raise RuntimeError("Temperature optimizer failed.")
    # T=1 is a valid candidate; do not keep a numerically worse solution.
    chosen = float(np.exp(result.x)) if result.fun <= objective(0.0) else 1.0
    return {"temperature": chosen, "calibration_nll_before": objective(0.0),
            "calibration_nll_after": objective(np.log(chosen)),
            "at_search_boundary": chosen < 0.051 or chosen > 19.9}

def grouped_accuracy_interval(logits, y, groups, seed=17, repetitions=500):
    # Cluster bootstrap by premise, rather than pretending all NLI pairs are independent.
    pred = np.asarray(logits).argmax(axis=1)
    y = np.asarray(y)
    distinct, inverse = np.unique(np.asarray(groups), return_inverse=True)
    correct = np.bincount(inverse, weights=(pred == y).astype(float))
    counts = np.bincount(inverse)
    rng = np.random.default_rng(seed)
    estimates = []
    for _ in range(repetitions):
        sampled = rng.integers(0, len(distinct), len(distinct))
        estimates.append(float(correct[sampled].sum() / counts[sampled].sum()))
    return {"low": float(np.percentile(estimates, 2.5)), "high": float(np.percentile(estimates, 97.5)),
            "method": "premise-cluster percentile bootstrap", "replicates": repetitions,
            "groups": len(distinct)}

# Known-value checks protect summary numbers from metric implementation mistakes.
uniform = np.zeros((3, 3), dtype=float)
assert abs(metric_values(uniform, [0, 1, 2])["nll"] - np.log(3)) < 1e-10
assert abs(metric_values(uniform, [0, 1, 2])["brier_sum_classes"] - 2/3) < 1e-10
assert np.allclose(probabilities(uniform).sum(axis=1), 1)
print("Metric unit checks passed. Multiclass Brier is SUM over classes, then mean over examples.")


## 12 · Train seven frozen-head variants

Fit linear/MLP heads on last, mean and max features; fit an attention+MLP head when enabled. All share the same examples.
Early stopping uses dev NLL. With multiple seeds, select the pooling/head configuration by **mean dev NLL**, and use the
first predeclared seed as the representative export—not the luckiest test seed. Head weights save as safetensors.

In [ ]:
from safetensors.torch import save_file, load_file

class CachedRows(Dataset):
    def __init__(self, store, pooling): self.store, self.pooling = store, pooling
    def __len__(self): return len(self.store)
    def __getitem__(self, idx):
        x = self.store.get_tokens(idx) if self.pooling == "attention" else np.array(self.store.fixed[self.pooling][idx], copy=True)
        return torch.from_numpy(x), int(self.store.labels[idx])

def collate_cached(items):
    xs, ys = zip(*items)
    if xs[0].ndim == 1:
        return {"x": torch.stack(xs), "labels": torch.tensor(ys)}
    lengths = [len(x) for x in xs]
    x = torch.zeros(len(xs), max(lengths), xs[0].shape[-1])
    mask = torch.zeros(len(xs), max(lengths), dtype=torch.bool)
    for i, item in enumerate(xs):
        x[i, :len(item)] = item; mask[i, :len(item)] = True
    return {"x": x, "mask": mask, "labels": torch.tensor(ys)}

def cached_loader(split, pooling, shuffle=False, seed=17):
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(CachedRows(STORES[split], pooling),
                      batch_size=CFG.attention_batch_size if pooling == "attention" else CFG.head_batch_size,
                      shuffle=shuffle, generator=generator, num_workers=0, collate_fn=collate_cached)

def head_logits(head, batch):
    x = batch["x"].to(DEVICE)
    return head(x, batch["mask"].to(DEVICE)) if "mask" in batch else head(x)

@torch.no_grad()
def predict_cached(head, split, pooling):
    head.eval()
    outputs = []
    for batch in cached_loader(split, pooling):
        z = head_logits(head, batch)
        if not torch.isfinite(z).all(): raise FloatingPointError("Non-finite prediction logits.")
        outputs.append(z.float().cpu().numpy())
    return np.concatenate(outputs)

MODELS, LOGITS, RUNS = {}, {}, {}

def fit_head(pooling, architecture, seed):
    seed_everything(seed)
    key = f"{pooling}_{architecture}_seed{seed}"
    head = AttentionHead(HIDDEN_SIZE, N_CLASSES, CFG.mlp_width) if pooling == "attention" else FixedHead(HIDDEN_SIZE, N_CLASSES, architecture, CFG.mlp_width)
    if pooling != "attention":
        x = np.asarray(STORES["train"].fixed[pooling], dtype=np.float32)
        # Train-only affine standardization leaves the linear probe a linear classifier.
        head.feature_mean.copy_(torch.from_numpy(x.mean(axis=0)))
        head.feature_std.copy_(torch.from_numpy(x.std(axis=0).clip(1e-3)))
    head = head.to(DEVICE)
    optimizer = torch.optim.AdamW(head.parameters(), lr=CFG.head_lr, weight_decay=CFG.head_weight_decay)
    best_nll, best_weights, stale, history = float("inf"), None, 0, []
    start = time.perf_counter()
    for epoch in range(CFG.head_epochs):
        head.train()
        sum_loss, seen = 0.0, 0
        for batch in cached_loader("train", pooling, shuffle=True, seed=seed + epoch):
            optimizer.zero_grad(set_to_none=True)
            z = head_logits(head, batch)
            y = batch["labels"].to(DEVICE)
            loss = F.cross_entropy(z, y)
            if not torch.isfinite(loss): raise FloatingPointError(f"Non-finite loss: {key}")
            loss.backward()
            torch.nn.utils.clip_grad_norm_(head.parameters(), 1.0)
            optimizer.step()
            sum_loss += loss.item() * len(y); seen += len(y)
        dev_logits = predict_cached(head, "dev", pooling)
        dev = metric_values(dev_logits, STORES["dev"].labels)
        history.append({"epoch": epoch + 1, "train_nll": sum_loss/seen, "dev_nll": dev["nll"], "dev_accuracy": dev["accuracy"]})
        if dev["nll"] < best_nll - 1e-5:
            best_nll = dev["nll"]
            best_weights = {k: v.detach().cpu().clone() for k, v in head.state_dict().items()}
            stale = 0
        else: stale += 1
        if stale >= CFG.head_patience: break
    if best_weights is None: raise RuntimeError("No valid head checkpoint.")
    head.load_state_dict(best_weights); head.eval()
    all_logits = {s: predict_cached(head, s, pooling) for s in ("dev", "calibration", "test_matched", "test_mismatched")}
    stats = {"key": key, "pooling": pooling, "head_architecture": architecture, "seed": seed,
             "parameters": sum(p.numel() for p in head.parameters()), "selection_dev_nll": best_nll,
             "epochs_run": len(history), "train_seconds": time.perf_counter()-start, "history": history}
    (OUT / "heads").mkdir(exist_ok=True)
    save_file({k: v.contiguous() for k, v in best_weights.items()}, str(OUT / "heads" / f"{key}.safetensors"))
    json_write(OUT / "heads" / f"{key}.json", stats)
    MODELS[key], LOGITS[key], RUNS[key] = head.cpu(), all_logits, stats
    del optimizer
    print(f"{key}: selected dev NLL={best_nll:.4f}; {len(history)} epochs")

for seed in CFG.head_seeds:
    for pooling in ("last", "mean", "max"):
        for architecture in ("linear", "mlp"):
            fit_head(pooling, architecture, seed)
    if CFG.run_attention_pooling:
        fit_head("attention", "mlp", seed)
STATUS["frozen"] = "completed"
json_write(OUT / "training_runs.json", RUNS)
# Select a configuration by mean DEV NLL across seeds. Do not choose by test accuracy.
config_scores = {}
for key, run in RUNS.items():
    config_scores.setdefault((run["pooling"], run["head_architecture"]), []).append(run["selection_dev_nll"])
best_config = min(config_scores, key=lambda c: np.mean(config_scores[c]))
WINNER = f"{best_config[0]}_{best_config[1]}_seed{CFG.head_seeds[0]}"
print("Configuration chosen on dev:", best_config, "; representative seed fixed in advance:", CFG.head_seeds[0])


## 13 · Evaluate untouched tests and compare baselines

Evaluate raw and calibrated probabilities on matched and mismatched tests. Include a smoothed train-label-prior baseline
and, when single-token codes exist, an **untuned restricted vocabulary projection** baseline. The latter normalizes only over
A/B/C token scores; it is not ground-truth confidence. Ordinary generation quality is not being inferred from these scores.

`neutral` is not silently relabeled `false`. NLI performance alone does not validate a general Binary/Choice/Score API.

In [ ]:
TEMPERATURES, RESULTS, RELIABILITY = {}, [], {}
# Two baselines: label base rates and a pretrained, finite-code vocabulary projection.
prior = np.bincount(STORES["train"].labels, minlength=N_CLASSES).astype(float) + 1.0
prior = prior / prior.sum()
LOGITS["train_prior"] = {s: np.tile(np.log(prior), (len(STORES[s]), 1))
                         for s in ("dev", "calibration", "test_matched", "test_mismatched")}
if CODE_WEIGHTS is not None:
    LOGITS["untuned_finite_code"] = {s: np.asarray(STORES[s].fixed["last"]) @ CODE_WEIGHTS.numpy().T
                                     for s in ("dev", "calibration", "test_matched", "test_mismatched")}
else:
    WARNINGS.append("Could not find single-token A/B/C codes; untuned finite-code baseline was skipped.")

PREDICTIONS = []
for name, split_logits in LOGITS.items():
    fitted = fit_temperature(split_logits["calibration"], STORES["calibration"].labels)
    TEMPERATURES[name] = fitted
    if fitted["at_search_boundary"]: WARNINGS.append(f"Temperature for {name} is near its search boundary.")
    for split in ("test_matched", "test_mismatched"):
        y = np.asarray(STORES[split].labels)
        logits = split_logits[split]
        ci = grouped_accuracy_interval(logits, y, [r["group"] for r in SPLITS[split]], CFG.seed)
        for calibrated in (False, True):
            temp = fitted["temperature"] if calibrated else 1.0
            metrics = metric_values(logits, y, temp)
            RESULTS.append({"run": name, "split": split, "calibrated": calibrated, **metrics,
                            "accuracy_ci_low": ci["low"], "accuracy_ci_high": ci["high"]})
            RELIABILITY[f"{name}/{split}/{calibrated}"] = reliability_bins(probabilities(logits, temp), y)
        p = probabilities(logits, fitted["temperature"])
        for idx, row in enumerate(SPLITS[split]):
            PREDICTIONS.append({"run": name, "split": split, "pair_hash": row["pair_hash"],
                                "label": LABELS[row["label"]], "prediction": LABELS[int(p[idx].argmax())],
                                **{f"p_{label}": float(p[idx, j]) for j, label in enumerate(LABELS)}})
METRICS_FRAME = pd.DataFrame(RESULTS)
METRICS_FRAME.to_csv(OUT / "metrics.csv", index=False)
pd.DataFrame(PREDICTIONS).to_csv(OUT / "predictions.csv", index=False)
json_write(OUT / "temperatures.json", TEMPERATURES)
json_write(OUT / "reliability_bins.json", RELIABILITY)
json_write(OUT / "metrics.json", RESULTS)
print("Calibrated test results (winner was selected using dev data, not this table):")
display(METRICS_FRAME[METRICS_FRAME.calibrated][["run", "split", "accuracy", "macro_f1", "nll", "brier_sum_classes", "ece_toplabel_15bins", "temperature"]].round(4))
print("Winning configuration:", WINNER)
print("Neutral remains a distinct NLI label. These are fixed NLI classes, not arbitrary Jev Choice/Score schemas.")


## 14 · Inspect accuracy and reliability curves

Inspect results, especially degradation on mismatched genres. Counts for each reliability bin are saved separately.
Small or empty bins are not evidence that a model is reliably calibrated. The figures are generated only after real evaluation.

In [ ]:
# No metric is inferred from synthetic training loss: all plots below use held-out results.
plots_dir = OUT / "plots"
plots_dir.mkdir(exist_ok=True)
plot_frame = METRICS_FRAME[(METRICS_FRAME.split == "test_matched") & METRICS_FRAME.calibrated]
fig, ax = plt.subplots(figsize=(10, max(4, len(plot_frame)*0.34)))
ax.barh(plot_frame.run, plot_frame.accuracy)
ax.set(xlim=(0, 1), xlabel="Held-out matched accuracy", title="Frozen decision heads and untuned baselines")
ax.grid(axis="x", alpha=0.25)
fig.tight_layout(); fig.savefig(plots_dir / "accuracy.png", dpi=150); plt.show(); plt.close(fig)

for split in ("test_matched", "test_mismatched"):
    fig, ax = plt.subplots(figsize=(6, 5))
    for calibrated in (False, True):
        bins = [r for r in RELIABILITY[f"{WINNER}/{split}/{calibrated}"] if r["count"]]
        ax.plot([r["confidence"] for r in bins], [r["accuracy"] for r in bins], marker="o",
                label="Temperature-scaled" if calibrated else "Uncalibrated")
    ax.plot([0, 1], [0, 1], linestyle="--", label="Perfect calibration reference")
    ax.set(xlim=(0, 1), ylim=(0, 1), xlabel="Mean predicted top-label probability", ylabel="Observed accuracy",
           title=f"{split}: {WINNER}")
    ax.legend(); ax.grid(alpha=0.2)
    fig.tight_layout(); fig.savefig(plots_dir / f"reliability_{split}.png", dpi=150); plt.show(); plt.close(fig)
print("Reliability curves describe these samples only. Per-bin sample counts are in reliability_bins.json.")


## 15 · Benchmark the learned decision path

Each timed call runs the **backbone**, not just a cached-vector classifier. Measure individual pooling variants, independent-example
batch throughput and a separate Python end-to-end path including tokenization, transfers and JSON serialization.
This does not include HTTP, Tokio scheduling, Rust kernels or shared-state question branching.
Allocated, reserved and incremental-peak GPU memory are different quantities; use the explicit labels.

In [ ]:
WINNING_HEAD = MODELS[WINNER].to(DEVICE).eval()
WINNING_POOLING = RUNS[WINNER]["pooling"]
WINNING_TEMPERATURE = TEMPERATURES[WINNER]["temperature"]

@torch.inference_mode()
def decision_forward(batch, head=WINNING_HEAD, pooling=WINNING_POOLING, network=None, temperature=WINNING_TEMPERATURE):
    hidden = forward_hidden(batch, network)
    mask = batch["attention_mask"].to(DEVICE)
    z = head(hidden, mask) if pooling == "attention" else head(pool_hidden(hidden, mask, pooling))
    return torch.softmax(z / temperature, dim=-1)

PERFORMANCE = []
# Keep only the head under test on the GPU during per-head measurements.
WINNING_HEAD.cpu()
# Same prompt as the generation benchmark, with every pooling method independently warmed.
for pooling in ("last", "mean", "max", "attention"):
    key = f"{pooling}_mlp_seed{CFG.head_seeds[0]}"
    if key not in MODELS: continue
    current_head = MODELS[key].to(DEVICE).eval()
    b = collate_tokens([BENCH_ROW])
    resident_b = {k: v.to(DEVICE) for k, v in b.items()}
    def run_this_head():
        return decision_forward(resident_b, current_head, pooling, temperature=TEMPERATURES[key]["temperature"])
    stats = timed_calls(run_this_head, CFG.benchmark_repeats, CFG.benchmark_warmups)
    stats.update({"mode": "decision", "run": key, "batch_size": 1, "input_tokens": BENCH_ROW["length"],
                  "decisions_per_second": 1000/stats["p50_ms"],
                  "scope": "GPU-resident input -> backbone -> pooling -> head -> softmax; excludes HTTP/tokenization"})
    PERFORMANCE.append(stats)
    current_head.cpu()
    del resident_b, b, run_this_head, current_head
# Winner scaling: throughput for independent examples, NOT shared-context question branching.
WINNING_HEAD.to(DEVICE)
for size in CFG.benchmark_batch_sizes:
    batch = collate_tokens(TOKENS["dev"][:size])
    resident = {k: v.to(DEVICE) for k, v in batch.items()}
    stats = timed_calls(lambda: decision_forward(resident), CFG.benchmark_repeats, CFG.benchmark_warmups)
    stats.update({"mode": "winner_batch", "run": WINNER, "batch_size": size,
                  "max_input_tokens": int(resident["input_ids"].shape[1]),
                  "decisions_per_second": size*1000/stats["p50_ms"],
                  "scope": "independent examples; no shared-prefill optimization"})
    PERFORMANCE.append(stats)
    del batch, resident
# A separate end-to-end Python path includes tokenization and the CUDA->CPU result transfer.
def end_to_end():
    row = encode_record(SPLITS["dev"][0])
    p = decision_forward(collate_tokens([row])).cpu().numpy()[0]
    return json.dumps({label: float(p[j]) for j, label in enumerate(LABELS)})
end_to_end_stats = timed_calls(end_to_end, CFG.benchmark_repeats, CFG.benchmark_warmups)
end_to_end_stats.update({"mode": "python_end_to_end", "run": WINNER, "batch_size": 1,
                        "scope": "tokenize -> host-to-device -> decision -> device-to-host -> JSON; no HTTP"})
PERFORMANCE.append(end_to_end_stats)
json_write(OUT / "decision_benchmark.json", PERFORMANCE)
display(pd.DataFrame([{k: r.get(k) for k in ("mode", "run", "batch_size", "p50_ms", "p95_ms", "decisions_per_second", "resident_allocated_mib", "peak_extra_allocated_mib")} for r in PERFORMANCE]).round(3))
print("p95 estimates from this small repeat count are descriptive, not production tail-latency guarantees.")
# Repeat alone vs padded batch at the probability level for the selected head.
with torch.inference_mode():
    joint = decision_forward(collate_tokens(TOKENS["dev"][:2])).cpu().numpy()
    separate = np.concatenate([decision_forward(collate_tokens([r])).cpu().numpy() for r in TOKENS["dev"][:2]])
BATCH_INVARIANCE = {"max_probability_difference": float(np.max(np.abs(joint-separate))), "examples": 2,
                    "meaning": "batch/padding numerical check, not shared-prefix branch isolation"}
json_write(OUT / "batch_invariance.json", BATCH_INVARIANCE)
print(BATCH_INVARIANCE)


## 16 · Export the frozen graph contract and parity fixtures

Save the chosen head, train-only normalization buffers, tokenizer, calibrated temperature and prompt/tokenization manifest.
Golden fixtures contain head inputs, logits and probabilities so a Rust head implementation can be compared numerically.
**The Qwen backbone weights are not copied into this export.** The manifest pins the original model commit. This is not a ready-made Rust backend.

In [ ]:
# Export a compact head-level Rust handoff BEFORE optional LoRA mutates the backbone.
EXPORT = OUT / "frozen_export"
EXPORT.mkdir(exist_ok=True)
WINNING_HEAD.to(DEVICE).eval()
save_file({k: v.detach().cpu().contiguous() for k, v in WINNING_HEAD.state_dict().items()}, str(EXPORT / "head.safetensors"))
manifest = {
    "format": "opendecision-fixed-nli-probe/v1", "model_id": CFG.model_id, "model_commit": MODEL_SHA,
    "text_backbone_class": ARCH["backbone_class"], "hidden_size": HIDDEN_SIZE,
    "head": RUNS[WINNER], "label_order": LABELS, "temperature": WINNING_TEMPERATURE,
    "dtype": str(DTYPE), "activation_cache_dtype": "float16", "pad_token_id": tokenizer.pad_token_id,
    "padding_side": "right", "pooling": WINNING_POOLING, "prompt_version": PROMPT_VERSION,
    "prompt_segments": {"prefix": PREFIX, "hypothesis_prefix": "\nHypothesis:\n", "tail": TAIL},
    "tokenization": "Tokenize each segment separately without special tokens, concatenate; follow encode_record for truncation.",
    "max_length": CFG.max_length, "head_input": "final text hidden states; masks exclude padding",
    "head_output": "three logits in label_order; probability = softmax(logits / temperature)",
    "normalization": "FixedHead contains train-only feature_mean/std buffers. MLP also has LayerNorm; GELU is exact.",
    "backbone_weights_included": False, "lora": False,
    "jev_dynamic_choices_implemented": False, "shared_state_cache_implemented": False,
    "parity_scope": "head.safetensors + golden_head_inputs.npz test the head only; not a Rust Qwen implementation",
}
json_write(EXPORT / "manifest.json", manifest)
tokenizer.save_pretrained(EXPORT / "tokenizer")
golden = {}
for i, row in enumerate(TOKENS["dev"][:3]):
    b = collate_tokens([row])
    with torch.no_grad():
        h = forward_hidden(b).float()
        mask = b["attention_mask"].to(DEVICE)
        x = h if WINNING_POOLING == "attention" else pool_hidden(h, mask, WINNING_POOLING)
        z = WINNING_HEAD(x, mask) if WINNING_POOLING == "attention" else WINNING_HEAD(x)
        p = torch.softmax(z / WINNING_TEMPERATURE, dim=-1)
    golden[f"input_{i}"] = x.cpu().numpy()
    golden[f"mask_{i}"] = mask.cpu().numpy()
    golden[f"logits_{i}"] = z.cpu().numpy()
    golden[f"probabilities_{i}"] = p.cpu().numpy()
    golden[f"token_ids_{i}"] = np.array(row["input_ids"], dtype=np.int64)
    del h, x, z, p, mask
np.savez_compressed(EXPORT / "golden_head_inputs.npz", **golden)
del golden
print("Frozen export:", EXPORT)
print("Includes tokenizer, head weights, calibration temperature, manifest and head-level parity vectors.")


## 17 · Optional: adapt the last hybrid blocks with LoRA

Skipped unless `run_lora=True`. Start from the selected frozen head, inspect actual `nn.Linear` modules in the configured last
blocks, and train with gradient accumulation plus non-reentrant checkpointing. The unquantized base remains frozen.

Epoch zero is a valid candidate: LoRA is not forced to replace a better frozen model. Checkpoint selection uses dev NLL;
calibration is refit separately. Exceptions are recorded without discarding frozen results. On GPUs without enough headroom
this cell reports **skipped**, not a fabricated comparison. This is one limited adaptation experiment, not an exhaustive tuning study.

In [ ]:
# Optional experiment: same data and prompt, but now adapt the last N hybrid blocks.
# This is LoRA (unquantized base), not QLoRA. Changing this to 4-bit would add another variable.
LORA_REPORT = {"status": STATUS["lora"], "reason": "Set CFG.run_lora=True in the settings cell to run this experiment."}
LORA_RESULTS = []

def run_lora_experiment():
    from peft import LoraConfig, get_peft_model
    if DTYPE != torch.bfloat16:
        return {"status": "skipped", "reason": "This optional training path requires a BF16-capable GPU; frozen evaluation still works with FP16."}
    free, _ = torch.cuda.mem_get_info(DEVICE)
    if free < 6 * 1024**3:
        return {"status": "skipped", "reason": "Less than 6 GiB GPU headroom. Use a fresh L4/A100-class allocation or fewer cached GPU tensors."}
    seed_everything(CFG.head_seeds[0])
    last_start = int(text_config.num_hidden_layers) - CFG.lora_last_layers
    if not 0 <= last_start < int(text_config.num_hidden_layers):
        raise ValueError("Invalid lora_last_layers.")
    target_names = []
    for name, module in backbone.named_modules():
        matched = re.match(r"layers\.(\d+)\.", name)
        if matched and int(matched.group(1)) >= last_start and isinstance(module, nn.Linear):
            target_names.append(name)
    if not target_names:
        raise RuntimeError("No actual nn.Linear modules found for LoRA.")
    conf = LoraConfig(r=CFG.lora_rank, lora_alpha=2*CFG.lora_rank, lora_dropout=0.05,
                      target_modules=target_names, bias="none")  # No generative task_type.
    network = get_peft_model(backbone, conf)
    network.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    network.config.use_cache = False
    # Warm start the same selected frozen head; compare adaptation, not random reinitialization.
    head = copy.deepcopy(MODELS[WINNER]).to(DEVICE)
    parameters = [p for p in network.parameters() if p.requires_grad] + list(head.parameters())
    optimizer = torch.optim.AdamW(parameters, lr=CFG.lora_lr, weight_decay=0.01)
    target_dir = OUT / "lora_export"
    target_dir.mkdir(exist_ok=True)
    history = []
    best_nll = RUNS[WINNER]["selection_dev_nll"]
    # Save the zero-update adapter too; training is not allowed to force selection of a worse epoch.
    network.save_pretrained(target_dir / "adapter", safe_serialization=True)
    save_file({k: v.detach().cpu().contiguous() for k, v in head.state_dict().items()}, str(target_dir / "head.safetensors"))
    selected_epoch, steps, seen = 0, 0, 0
    def train_logits(batch):
        hidden = forward_hidden(batch, network)
        mask = batch["attention_mask"].to(DEVICE)
        return head(hidden, mask) if WINNING_POOLING == "attention" else head(pool_hidden(hidden, mask, WINNING_POOLING))
    @torch.no_grad()
    def evaluate_live(split):
        head.eval(); network.eval()
        all_z = []
        loader = DataLoader(TokenRows(TOKENS[split]), batch_size=CFG.feature_batch_size, shuffle=False,
                            num_workers=0, collate_fn=collate_tokens)
        for batch in loader:
            z = train_logits(batch)
            if not torch.isfinite(z).all(): raise FloatingPointError("Non-finite LoRA evaluation logits.")
            all_z.append(z.float().cpu().numpy())
        return np.concatenate(all_z)
    started = time.perf_counter()
    torch.cuda.reset_peak_memory_stats(DEVICE)
    for epoch in range(CFG.lora_epochs):
        loader = DataLoader(TokenRows(TOKENS["train"]), batch_size=CFG.lora_batch_size,
                            shuffle=True, generator=torch.Generator().manual_seed(CFG.seed+epoch),
                            num_workers=0, collate_fn=collate_tokens)
        network.train(); head.train(); optimizer.zero_grad(set_to_none=True)
        loss_sum, epoch_seen, accumulated_examples = 0.0, 0, 0
        for step, batch in enumerate(tqdm(loader, desc=f"LoRA epoch {epoch+1}")):
            z = train_logits(batch)
            y = batch["labels"].to(DEVICE)
            # Sum losses and divide gradients by actual examples per update, including the final short window.
            loss = F.cross_entropy(z, y, reduction="sum")
            if not torch.isfinite(loss): raise FloatingPointError("Non-finite LoRA loss.")
            loss.backward()
            loss_sum += loss.item(); epoch_seen += len(y); seen += len(y); accumulated_examples += len(y)
            if (step+1) % CFG.lora_accumulation == 0 or step+1 == len(loader):
                for p in parameters:
                    if p.grad is not None: p.grad.div_(accumulated_examples)
                torch.nn.utils.clip_grad_norm_(parameters, 1.0)
                optimizer.step(); optimizer.zero_grad(set_to_none=True)
                accumulated_examples = 0; steps += 1
            del z, y, loss
        dev_z = evaluate_live("dev")
        dev_metrics = metric_values(dev_z, STORES["dev"].labels)
        history.append({"epoch": epoch+1, "train_nll": loss_sum/epoch_seen, "dev": dev_metrics})
        if dev_metrics["nll"] < best_nll:
            best_nll = dev_metrics["nll"]; selected_epoch = epoch+1
            network.save_pretrained(target_dir / "adapter", safe_serialization=True)
            save_file({k: v.detach().cpu().contiguous() for k, v in head.state_dict().items()}, str(target_dir / "head.safetensors"))
        print(f"LoRA epoch {epoch+1}: dev NLL {dev_metrics['nll']:.4f}; best epoch {selected_epoch}")
    training_peak = torch.cuda.max_memory_allocated(DEVICE) / 1024**2
    training_seconds = time.perf_counter() - started
    # Restore the selected adapter without constructing another 4B backbone.
    from peft.utils.save_and_load import load_peft_weights, set_peft_model_state_dict
    adapter_weights = load_peft_weights(str(target_dir / "adapter"), device="cpu")
    set_peft_model_state_dict(network, adapter_weights, adapter_name="default")
    head.load_state_dict(load_file(str(target_dir / "head.safetensors")))
    del optimizer, parameters, adapter_weights
    network.gradient_checkpointing_disable()
    network.eval(); head.eval()
    logits = {s: evaluate_live(s) for s in ("dev", "calibration", "test_matched", "test_mismatched")}
    temp = fit_temperature(logits["calibration"], STORES["calibration"].labels)
    for split in ("test_matched", "test_mismatched"):
        for calibrated in (False, True):
            LORA_RESULTS.append({"run": "lora_selected", "split": split, "calibrated": calibrated,
                                 **metric_values(logits[split], STORES[split].labels, temp["temperature"] if calibrated else 1.0)})
    batch = {k: v.to(DEVICE) for k, v in collate_tokens([BENCH_ROW]).items()}
    benchmark = timed_calls(lambda: decision_forward(batch, head, WINNING_POOLING, network, temp["temperature"]),
                            CFG.benchmark_repeats, CFG.benchmark_warmups)
    report = {"status": "completed", "selected_epoch": selected_epoch, "best_dev_nll": best_nll,
              "training_seconds": training_seconds, "optimizer_steps": steps, "examples_seen": seen,
              "last_layers": CFG.lora_last_layers, "target_modules": target_names,
              "adapter_trainable_parameters": sum(p.numel() for p in network.parameters() if p.requires_grad),
              "peak_training_allocated_mib": training_peak, "temperature": temp,
              "history": history, "metrics": LORA_RESULTS, "benchmark": benchmark,
              "caveat": "Single-seed warm-started last-block LoRA experiment; not an exhaustive fine-tuning search."}
    json_write(target_dir / "manifest.json", {**manifest, "lora": True, "adapter_directory": "adapter", "lora_report": report,
                                              "temperature": temp["temperature"], "parity_scope": "No LoRA golden vectors were exported."})
    json_write(target_dir / "metrics.json", LORA_RESULTS)
    return report

if CFG.run_lora:
    # Optional failures are explicit, but cannot erase the completed frozen results.
    try:
        LORA_REPORT = run_lora_experiment()
        STATUS["lora"] = LORA_REPORT["status"]
    except Exception as error:
        # Preserve an actionable message, with any literal token or credential-shaped string redacted.
        safe_message = str(error)
        if HF_TOKEN:
            safe_message = safe_message.replace(HF_TOKEN, "[REDACTED]")
        safe_message = re.sub(r"hf_[A-Za-z0-9_]+", "[REDACTED]", safe_message)
        LORA_REPORT = {"status": "failed", "error_type": type(error).__name__,
                       "message": safe_message[:600], "frozen_results_preserved": True}
        STATUS["lora"] = "failed"
        WARNINGS.append(f"LoRA failed ({type(error).__name__}); no successful LoRA comparison should be inferred.")
        gc.collect(); torch.cuda.empty_cache()
json_write(OUT / "lora_report.json", LORA_REPORT)
print(json.dumps({k: v for k, v in LORA_REPORT.items() if k not in ("target_modules", "history")}, indent=2))


## 18 · Print the summary to send back

The compact JSON between the `BEGIN_OPENDECISION_PHASE2B_SUMMARY` and `END` markers is designed to paste back into chat.
The complete JSON retains all metrics, timings, provenance and limitations. The Markdown readout is generated from measured values.
Do not turn the generation/decision ratio into an equal-quality speedup claim.

In [ ]:
# All conclusions below are generated from measured results. Nothing is filled with expected numbers.
def matched_result(run, split="test_matched", calibrated=True):
    return next(r for r in RESULTS if r["run"] == run and r["split"] == split and r["calibrated"] == calibrated)

selected_matched = matched_result(WINNER)
selected_mismatched = matched_result(WINNER, "test_mismatched")
plain_matched = matched_result(WINNER, calibrated=False)
selected_rows = [r for r in PERFORMANCE if r["mode"] == "winner_batch" and r["batch_size"] == 1]
selected_timing = selected_rows[0] if selected_rows else None
ratios = [{"generation_budget": r["max_new_tokens"], "actual_new_tokens": r["actual_new_tokens"],
           "generation_p50_ms": r["p50_ms"], "decision_p50_ms": selected_timing["p50_ms"],
           "latency_ratio": r["p50_ms"] / selected_timing["p50_ms"],
           "caveat": "Different output workloads; not equal-quality speedup."}
          for r in GENERATION_RESULTS] if selected_timing else []

LIMITATIONS = [
    "This is a fixed three-class NLI experiment, not a trained arbitrary-schema Jev replacement.",
    "No shared-prefix cache fork, inter-question isolation architecture, dynamic candidate head, or ordinal Score training is implemented.",
    "The pretraining corpus may contain MultiNLI. Split separation here does not establish decontamination of Qwen pretraining.",
    "Removing a tied LM head does not remove the input embedding or most text parameters.",
    "No universal calibration or production latency guarantee follows from these sampled tasks and short timing runs.",
    "Generation timings and classification timings do not demonstrate equal decision quality.",
    "Attention pooling can differ because of extra trainable parameters, not only pooling choice.",
    "No quantization, Rust, Metal, or cross-device parity benchmark was run by this notebook.",
]
if len(CFG.head_seeds) == 1:
    LIMITATIONS.append("Only one head-training seed was evaluated; accuracy intervals quantify test sampling, not training-seed uncertainty.")
if not CFG.run_lora:
    LIMITATIONS.append("LoRA was disabled; this run cannot conclude whether LoRA improves the backbone.")

SUMMARY = {
    "schema": "opendecision-phase2b-summary/v1", "run_id": RUN_ID, "status": STATUS,
    "model": CFG.model_id, "model_commit": MODEL_SHA, "dataset_commit": DATA_SHA,
    "environment": ENV, "architecture": ARCH, "data": DATA_AUDIT, "tokenization": TOKEN_STATS,
    "configuration": asdict(CFG), "selected_by": "lowest mean uncalibrated dev NLL across configured seeds",
    "winner": WINNER, "matched_test": selected_matched, "mismatched_test": selected_mismatched,
    "matched_uncalibrated": plain_matched, "temperature_fit": TEMPERATURES[WINNER],
    "all_metrics": RESULTS, "performance": PERFORMANCE, "generation_comparison": ratios,
    "batch_invariance": BATCH_INVARIANCE, "feature_extraction": CACHE_TIMES,
    "lora": LORA_REPORT, "warnings": WARNINGS, "limitations": LIMITATIONS,
    "source_notebook": SOURCE_NOTEBOOK["url"], "frozen_export": str(EXPORT),
}
json_write(OUT / "opendecision_phase2b_summary.json", SUMMARY)
calibration_delta = selected_matched["nll"] - plain_matched["nll"]
readout = f"""# OpenDecision Phase 2B — measured readout

Run: {RUN_ID}  
Model: {CFG.model_id} @ {MODEL_SHA}  
GPU: {ENV['gpu']}; dtype: {DTYPE}

## Selection and quality
Selected on **dev NLL**: `{WINNER}`.

| Evaluation | Accuracy | Macro F1 | NLL | Brier (sum) | ECE (15 bins) |
|---|---:|---:|---:|---:|---:|
| Matched, raw | {plain_matched['accuracy']:.4f} | {plain_matched['macro_f1']:.4f} | {plain_matched['nll']:.4f} | {plain_matched['brier_sum_classes']:.4f} | {plain_matched['ece_toplabel_15bins']:.4f} |
| Matched, calibrated | {selected_matched['accuracy']:.4f} | {selected_matched['macro_f1']:.4f} | {selected_matched['nll']:.4f} | {selected_matched['brier_sum_classes']:.4f} | {selected_matched['ece_toplabel_15bins']:.4f} |
| Mismatched, calibrated | {selected_mismatched['accuracy']:.4f} | {selected_mismatched['macro_f1']:.4f} | {selected_mismatched['nll']:.4f} | {selected_mismatched['brier_sum_classes']:.4f} | {selected_mismatched['ece_toplabel_15bins']:.4f} |

Matched accuracy 95% premise-cluster bootstrap interval: [{selected_matched['accuracy_ci_low']:.4f}, {selected_matched['accuracy_ci_high']:.4f}].
Temperature: {WINNING_TEMPERATURE:.4f}, fitted on a separate calibration set.
Matched test NLL change after calibration: {calibration_delta:+.4f} (negative is better; no guarantee under shift).

## Model and resources
Text-backbone parameters: {ARCH['backbone_after_detach']['parameters']:,}.
Logical vocabulary head: {ARCH['logical_lm_head_parameters']:,}; marginal removable parameters: {ARCH['marginal_lm_head_parameters']:,}.
Input/output weights tied: {TIED}. Vision tower loaded: {ARCH['vision_tower_loaded']}.

Frozen run: {STATUS['frozen']}. LoRA: {STATUS['lora']}.
This run does not choose a production architecture automatically; compare quality, calibration and latency together.

## Limitations
""" + "\n".join("- " + item for item in LIMITATIONS + WARNINGS)
(OUT / "opendecision_phase2b_summary.md").write_text(readout, encoding="utf-8")
display(Markdown(readout))

# Smaller paste-back report; the full JSON retains all experiments and audit details.
COMPACT = {
    "schema": SUMMARY["schema"], "run_id": RUN_ID, "status": STATUS,
    "model": CFG.model_id, "commit": MODEL_SHA, "gpu": ENV["gpu"], "dtype": str(DTYPE),
    "packages": ENV["packages"], "backbone_class": ARCH["backbone_class"],
    "backbone_parameters": ARCH["backbone_after_detach"]["parameters"],
    "marginal_lm_head_parameters": ARCH["marginal_lm_head_parameters"], "weights_tied": TIED,
    "split_sizes": {s: len(rows) for s, rows in SPLITS.items()}, "winner": WINNER,
    "matched": selected_matched, "mismatched": selected_mismatched,
    "head_results": [{k: r[k] for k in ("run", "split", "accuracy", "nll", "brier_sum_classes", "ece_toplabel_15bins")}
                     for r in RESULTS if r["calibrated"]],
    "timings": [{k: r.get(k) for k in ("mode", "run", "batch_size", "p50_ms", "p95_ms", "resident_allocated_mib", "peak_extra_allocated_mib")} for r in PERFORMANCE],
    "generation_ratios_not_equal_quality": ratios, "batch_invariance": BATCH_INVARIANCE,
    "lora_status": LORA_REPORT["status"], "lora_metrics": LORA_RESULTS,
    "warnings": WARNINGS, "limitations": LIMITATIONS,
}
json_write(OUT / "paste_back_summary.json", COMPACT)
print("\n===== BEGIN_OPENDECISION_PHASE2B_SUMMARY =====")
print(json.dumps(COMPACT, indent=2, allow_nan=False))
print("===== END_OPENDECISION_PHASE2B_SUMMARY =====")


## 19 · Preserve the results before Colab disconnects

The results ZIP contains reports, figures, head exports and optional LoRA adapters—not the multi-GiB feature cache or any token.
With Drive saving enabled, authorize the mount and find the archive under
`MyDrive/Colab Notebooks/OpenDecision_Phase2B_results/<run_id>/`.
The notebook itself has been created separately; running this cell does not overwrite your original probe.

In [ ]:
# A completed archive excludes the multi-gigabyte activation cache and all secrets.
archive_path = Path(shutil.make_archive(str(ROOT / f"opendecision_phase2b_{RUN_ID}"), "zip", OUT))
print("Local results:", OUT)
print("Result archive:", archive_path)
DRIVE_SAVE = {"status": "disabled"}
if CFG.copy_results_to_drive:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        destination = Path("/content/drive/MyDrive/Colab Notebooks/OpenDecision_Phase2B_results") / RUN_ID
        destination.mkdir(parents=True, exist_ok=True)
        for filename in ("opendecision_phase2b_summary.json", "opendecision_phase2b_summary.md", "paste_back_summary.json", "metrics.csv"):
            shutil.copy2(OUT / filename, destination / filename)
        shutil.copy2(archive_path, destination / archive_path.name)
        DRIVE_SAVE = {"status": "saved", "path": str(destination)}
        print("Copied reports and archive to:", destination)
    except Exception as error:
        DRIVE_SAVE = {"status": "failed", "error_type": type(error).__name__,
                      "message": "Results remain in the local output directory. Use the links below before the runtime disconnects."}
        print(DRIVE_SAVE)
json_write(OUT / "drive_save_status.json", DRIVE_SAVE)
from IPython.display import FileLink
for path in (OUT / "paste_back_summary.json", OUT / "opendecision_phase2b_summary.json", archive_path):
    display(FileLink(str(path)))
print("Send back paste_back_summary.json or the printed BEGIN/END summary block. No HF_TOKEN is included.")


## How to interpret the next decision

A low training loss alone is not success. Compare the **linear versus MLP** matched-test results, then the same head on mismatched
genres. Inspect calibration NLL/Brier before and after temperature scaling and the selective errors that remain.

A strong frozen probe justifies refining the prompt/data and testing genuinely dynamic candidate scoring next.
A weaker probe motivates the optional LoRA experiment, a different representation, or a more capable backbone; it does not prove
that generation is necessary. A useful NLI score is still not proof of Jev-compatible zero-shot semantics.

For follow-up review, send **`paste_back_summary.json`**. For debugging, include **`environment.json`**, **`architecture.json`**
and any failing cell output; do not send `HF_TOKEN`.
